# LLM Laika
Antes de rodar: menu **Ambiente de execução → Alterar tipo → GPU (T4)**.

In [ ]:
# 1) Clonar/atualizar, ligar o Google Drive e baixar os livros
# Livros e checkpoints ficam no seu Drive (pasta LLM_Laika), então sobrevivem se a sessão cair.
# Pode rodar de novo à vontade: não apaga nada.
import os, torch
print("GPU disponível:", torch.cuda.is_available())

from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/LLM_Laika"
os.makedirs(f"{DRIVE}/books", exist_ok=True)
os.makedirs(f"{DRIVE}/checkpoints", exist_ok=True)

os.chdir("/content")
if not os.path.exists("LLM_Laika"):
    !git clone https://github.com/BrunoDevPro/LLM_Laika.git
%cd /content/LLM_Laika
!git pull -q
!pip install -q tokenizers

# data/books e checkpoints viram atalhos para o Drive
for local, remote in [("data/books", f"{DRIVE}/books"), ("checkpoints", f"{DRIVE}/checkpoints")]:
    if not os.path.islink(local):
        if os.path.isdir(local) and os.listdir(local):
            !cp -rn {local}/. {remote}/
        !rm -rf {local}
        os.symlink(remote, local)

!python prepare_data.py --max-books 400

In [ ]:
# 2) Pré-treino (aprende português) + fine-tuning (aprende a responder)
# Se já existe pré-treino no Drive, ele é reaproveitado (não treina de novo).
# Para forçar um pré-treino novo, apague checkpoints/pretrain_best.pt no Drive.
import os
if os.path.exists("checkpoints/pretrain_best.pt"):
    print("Pré-treino encontrado no Drive: pulando.")
else:
    !python train.py --preset colab
!python finetune.py

In [ ]:
# 3) Conversar (digite 'sair' para encerrar). Para testar só o pré-treino: sys.argv = ["chat.py", "--base"]
import sys
sys.argv = ["chat.py"]
%run chat.py